<a href="https://colab.research.google.com/github/vanashri-18/CSA6101-Digital-Forensics-and-Cybercrime-Investigation/blob/main/DoS_Traffic_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

***Aim***

To develop a Python program that analyzes simulated packet-level network records to identify patterns resembling denial-of-service (DoS) behavior by examining packet frequency, source distribution, destination targets, and time intervals, while distinguishing concentrated traffic from normal distributed communication.

**Algorithm**

Create a simulated packet-level dataset.

Read timestamp, source IP, destination IP, destination port, and packet size.

Calculate the total packet count.

Group packets by destination host/service.

Calculate the number of unique sources contacting each destination.

Calculate packet frequency within the observed time period.

Examine the time interval between consecutive packets.

Identify destinations receiving unusually concentrated traffic.

Compare concentrated traffic with normal distributed communication.

Display the evidence and detected patterns.

Report the result as potentially suspicious behavior, not proof of an actual attack.

In [1]:
# ==============================================
# DOS-LIKE TRAFFIC ANALYZER
# ==============================================

import pandas as pd

# ------------------------------------------------
# 1. Simulated Packet Records
# ------------------------------------------------

data = [
    ["10:00:01", "192.168.1.10", "10.0.0.50", 80],
    ["10:00:02", "192.168.1.11", "10.0.0.50", 80],
    ["10:00:03", "192.168.1.12", "10.0.0.50", 80],
    ["10:00:04", "192.168.1.13", "10.0.0.50", 80],
    ["10:00:05", "192.168.1.14", "10.0.0.50", 80],
    ["10:00:06", "192.168.1.15", "10.0.0.50", 80],
    ["10:00:07", "192.168.1.10", "10.0.0.50", 80],
    ["10:00:08", "192.168.1.11", "10.0.0.50", 80],
    ["10:00:09", "192.168.1.12", "10.0.0.50", 80],
    ["10:00:10", "192.168.1.13", "10.0.0.50", 80],

    # Normal distributed traffic
    ["10:01:00", "192.168.2.10", "10.0.0.60", 443],
    ["10:02:00", "192.168.2.11", "10.0.0.61", 443],
    ["10:03:00", "192.168.2.12", "10.0.0.62", 443],
    ["10:04:00", "192.168.2.13", "10.0.0.63", 443],
    ["10:05:00", "192.168.2.14", "10.0.0.64", 443]
]

df = pd.DataFrame(
    data,
    columns=[
        "Timestamp",
        "Source_IP",
        "Destination_IP",
        "Destination_Port"
    ]
)

# Convert timestamp
df["Timestamp"] = pd.to_datetime(
    df["Timestamp"],
    format="%H:%M:%S"
)

print("=" * 95)
print("                    DOS-LIKE TRAFFIC ANALYZER")
print("=" * 95)

# ------------------------------------------------
# 2. Investigator Thresholds
# ------------------------------------------------

packet_threshold = int(
    input(
        "\nEnter packet threshold for a destination: "
    )
)

source_threshold = int(
    input(
        "Enter minimum unique-source threshold: "
    )
)

# ------------------------------------------------
# 3. Destination Statistics
# ------------------------------------------------

destination_stats = (
    df.groupby(
        ["Destination_IP", "Destination_Port"]
    )
    .agg(
        Packet_Count=("Source_IP", "count"),
        Unique_Sources=("Source_IP", "nunique")
    )
    .reset_index()
)

# ------------------------------------------------
# 4. Calculate Packet Rate
# ------------------------------------------------

start_time = df["Timestamp"].min()
end_time = df["Timestamp"].max()

duration = (
    end_time - start_time
).total_seconds()

if duration == 0:
    duration = 1

total_packets = len(df)

overall_rate = (
    total_packets / duration
)

# ------------------------------------------------
# 5. Detect Concentrated Traffic
# ------------------------------------------------

patterns = []

for _, row in destination_stats.iterrows():

    reasons = []

    if row["Packet_Count"] >= packet_threshold:

        reasons.append(
            "High packet frequency"
        )

    if row["Unique_Sources"] >= source_threshold:

        reasons.append(
            "Large source distribution"
        )

    # Determine concentration
    percentage = (
        row["Packet_Count"]
        / total_packets
    ) * 100

    if percentage >= 50:

        reasons.append(
            f"Traffic concentrated on this target "
            f"({percentage:.1f}% of packets)"
        )

    if reasons:

        patterns.append({
            "Destination": row["Destination_IP"],
            "Port": row["Destination_Port"],
            "Packets": row["Packet_Count"],
            "Unique_Sources": row["Unique_Sources"],
            "Traffic_Percentage": round(
                percentage, 2
            ),
            "Evidence": "; ".join(reasons)
        })

# ------------------------------------------------
# 6. Display Destination Statistics
# ------------------------------------------------

print("\n" + "=" * 95)
print("                    DESTINATION STATISTICS")
print("=" * 95)

print(
    destination_stats.to_string(
        index=False
    )
)

# ------------------------------------------------
# 7. Display Detected Patterns
# ------------------------------------------------

print("\n" + "=" * 95)
print("                POTENTIALLY SUSPICIOUS PATTERNS")
print("=" * 95)

if patterns:

    pattern_df = pd.DataFrame(patterns)

    print(
        pattern_df.to_string(
            index=False
        )
    )

else:

    print(
        "No concentrated traffic pattern detected."
    )

# ------------------------------------------------
# 8. Time Interval Analysis
# ------------------------------------------------

df_sorted = df.sort_values(
    "Timestamp"
)

intervals = (
    df_sorted["Timestamp"]
    .diff()
    .dt.total_seconds()
    .dropna()
)

average_interval = intervals.mean()

print("\n" + "=" * 95)
print("                     TIME ANALYSIS")
print("=" * 95)

print(
    "Average interval between packets :",
    round(average_interval, 2),
    "seconds"
)

print(
    "Overall packet rate              :",
    round(overall_rate, 2),
    "packets/second"
)

# ------------------------------------------------
# 9. Normal vs Concentrated Traffic
# ------------------------------------------------

print("\n" + "=" * 95)
print("                  TRAFFIC DISTRIBUTION")
print("=" * 95)

largest = destination_stats.sort_values(
    "Packet_Count",
    ascending=False
).iloc[0]

if largest["Packet_Count"] / total_packets >= 0.50:

    print(
        "Pattern: CONCENTRATED TRAFFIC"
    )

    print(
        f"{largest['Packet_Count']} of "
        f"{total_packets} packets target "
        f"{largest['Destination_IP']}."
    )

else:

    print(
        "Pattern: DISTRIBUTED TRAFFIC"
    )

    print(
        "Traffic is distributed across "
        "multiple destination hosts."
    )

# ------------------------------------------------
# 10. Final Interpretation
# ------------------------------------------------

print("\n" + "=" * 95)
print("                       INTERPRETATION")
print("=" * 95)

if patterns:

    print(
        "Potential DoS-like traffic patterns "
        "were detected."
    )

    print(
        "The activity shows unusually concentrated "
        "traffic toward one or more services."
    )

    print(
        "These observations require further "
        "investigation and do not alone prove "
        "a denial-of-service attack."
    )

else:

    print(
        "No strong DoS-like pattern was observed."
    )

print("\nAnalysis completed.")
print("=" * 95)

                    DOS-LIKE TRAFFIC ANALYZER

Enter packet threshold for a destination: 5
Enter minimum unique-source threshold: 5

                    DESTINATION STATISTICS
Destination_IP  Destination_Port  Packet_Count  Unique_Sources
     10.0.0.50                80            10               6
     10.0.0.60               443             1               1
     10.0.0.61               443             1               1
     10.0.0.62               443             1               1
     10.0.0.63               443             1               1
     10.0.0.64               443             1               1

                POTENTIALLY SUSPICIOUS PATTERNS
Destination  Port  Packets  Unique_Sources  Traffic_Percentage                                                                                                 Evidence
  10.0.0.50    80       10               6               66.67 High packet frequency; Large source distribution; Traffic concentrated on this target (66.7% of packets

**Result**

The program successfully analyzed the simulated packet records and identified concentrated traffic toward 10.0.0.50:80. The destination received 10 of 15 packets from 6 different sources, while the remaining traffic was distributed across different hosts. The combination of high packet frequency, multiple sources, and concentration on one service was reported as a potentially suspicious DoS-like pattern. The program intentionally treats this as an indicator requiring further investigation, not as proof of an actual attack.